# Region of attraction in three dimensions: four Kuramoto oscillators

With four oscillators the reduced state has dimension 3, and the certified
region cannot be drawn directly. This notebook computes it with `verify_roa` and
draws cross-sections with `plot_roa_slice`.

Requirements: Python 3.10 or later and
`pip install "pyddrv[jax,examples] @ git+https://github.com/NetDLab/pyDDRV"`.
See `examples/notebooks/README.md`.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_roa
from pyddrv.systems.fields_jax import kuramoto_reduced_jax
from pyddrv.viz import plot_roa_slice

## Computing the region

The settings follow the three-oscillator notebook. To keep the run short, only
the first pass is run (`trim=False`) and the time is limited to 90 seconds. A
result meant to be reported should use `trim=True`.

In [ ]:
k, n = 10.0, 4
f, L_bound = kuramoto_reduced_jax(k=k, n=n)
R, d = np.pi, n - 1                       # d = 3

t0 = time.time()
roa = verify_roa(f, R=R, d=d, alpha=1.0, L=L_bound, norm="inf",
                 tau=1.9, eps=np.pi/27, max_refine=4, max_seconds=90)
print(roa.summary())
print(f"certified fraction of Q_R: {roa.volume/(2*R)**d:.1%}, "
      f"wall {time.time()-t0:.0f}s")

About 66% of the box is accepted by the first pass, with around a million
cubes.
`roa.centers` now has three columns.

## Cross-sections

`plot_roa_slice(roa, dims, at)` draws the cubes that intersect the plane on
which the coordinates not listed in `dims` take the values given in `at`. A cube
with center `c` and half-width `h` intersects the plane when `|c_j − at_j| ≤ h`
for every fixed coordinate `j`. By default the plane passes through the
equilibrium.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, phi3 in zip(axes, [0.0, 1.2, 2.4]):
    plot_roa_slice(roa, dims=(0, 1), at=[0.0, 0.0, phi3], ax=ax)
    ax.set_xlabel(r"$\phi_1$"); ax.set_ylabel(r"$\phi_2$")
    ax.set_title(rf"slice at $\phi_3 = {phi3:g}$")
fig.suptitle(f"Kuramoto n={n} (d=3): slices of the certified 1-RoA", y=1.02)
fig.tight_layout()
plt.show()

In the plane `φ3 = 0` the section resembles the three-oscillator region,
including the excluded ball at the origin. At `φ3 = 1.2` the plane no longer
meets that ball, and the section is smaller and shifted. At `φ3 = 2.4` only a
small part remains. The area of a section is not the volume of the region,
which is `roa.volume`.